### Cell 1: 라이브러리 임포트 및 기본 설명

In [12]:
# -*- coding: utf-8 -*-
"""
페르소나 JSON + 제품 CSV -> 구매예측용 '최적 프롬프트' 자동 생성 스크립트
- 입력:
  - PERSONA_JSON: 2200명 페르소나 (예: /mnt/data/persona_core_summary_fixed (1).json)
  - PRODUCTS_CSV: 신제품 정보 (예: /mnt/data/product_info_v2 (1).csv)
- 출력:
  - build_optimal_prompt(persona, products, ...) -> str
  - (옵션) batch로 모든 페르소나 프롬프트 파일 저장
"""

import json
import re
import math
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any
from textwrap import dedent

### Cell 2: 경로 설정

In [13]:
# ===== 0. 경로 설정 =====
# 사용자 파일 경로에 맞게 변경하세요.
# 예시: "data/persona_core_summary_fixed (1).json"
PERSONA_JSON = "persona.json"

# 예시: "data/product_info_v2 (1).csv"
PRODUCTS_CSV = "product_info.csv" # 이 파일은 제가 가지고 있지 않으므로, 사용자께서 올바른 경로를 지정해주셔야 합니다.

### Cell 3: 유틸리티 함수
- 데이터를 안전하게 처리하고 변환하는 데 사용되는 함수

In [14]:
# ===== 1. 유틸 =====
def _safe_get(d: dict, path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict): 
            return default
        cur = cur.get(k)
        if cur is None:
            return default
    return cur

def _to_float(x, default=None):
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return default
        return float(str(x).replace(",", "").strip())
    except:
        return default

def _norm_1to9(x, default=0.5):
    """1~9 점수를 0~1로 정규화. 소수/결측 방어."""
    v = _to_float(x, None)
    if v is None: 
        return default
    v = max(1.0, min(9.0, v))
    return (v - 1.0) / 8.0

### Cell 4: 구매주기 텍스트 처리 함수
- '주 2~3회' 와 같은 텍스트 형식의 구매주기를 월간 구매 빈도수(숫자)로 변환

In [15]:
# ===== 2. 구매주기 텍스트 -> 월간 빈도 추정 =====
_PURCHASE_PATTERNS = [
    # "주 2~3회" "주 3회" 등
    (re.compile(r"주\s*(\d+)\s*~\s*(\d+)\s*회"), lambda a,b: ((int(a)+int(b))/2.0) * 4.3),
    (re.compile(r"주\s*(\d+)\s*회"), lambda a: int(a)*4.3),
    # "2주일에 1회" "2주 1회"
    (re.compile(r"(\d+)\s*주\s*일?\s*에\s*1\s*회"), lambda a: 4.3/float(a)),
    (re.compile(r"(\d+)\s*주\s*\/?\s*1\s*회"), lambda a: 4.3/float(a)),
    # "1달에 1회" "한달 1회" "월 1회"
    (re.compile(r"(?:1|한)\s*달\s*에\s*1\s*회"), lambda : 1.0),
    (re.compile(r"월\s*(\d+)\s*회"), lambda a: float(a)),
    # "분기 1회" 대략 0.33/월
    (re.compile(r"분기\s*1\s*회"), lambda : 1.0/3.0),
]

def purchase_cycle_to_monthly_freq(text: str, fallback: float=1.0) -> float:
    if not text:
        return fallback
    s = str(text).strip()
    for pat, fn in _PURCHASE_PATTERNS:
        m = pat.search(s)
        if m:
            try:
                return float(fn(*m.groups()))
            except:
                continue
    # 키워드 휴리스틱
    if "가끔" in s or "드묾" in s:
        return 0.5
    if "자주" in s:
        return 2.0
    return fallback

### Cell 5: 데이터 로딩 함수
- 제품(CSV) 및 페르소나(JSON) 파일을 불러와 파이썬에서 사용할 수 있는 형태로 변환

In [16]:
# ===== 3. 제품 데이터 로드 & 스키마 보강 =====
def load_products(path: str) -> List[Dict[str, Any]]:
    df = pd.read_csv(path)
    # product_name에 용량이 함께 온다고 가정 -> g/ml 추출(없어도 동작)
    size_ml = []
    for name in df["product_name"].astype(str):
        m = re.search(r"(\d+)\s*(g|ml|mL|G|ML)", name)
        size_ml.append(m.group(1)+m.group(2).lower() if m else "")
    df["size"] = size_ml

    # 권장 추가필드(없는 경우 placeholder)
    for col in ["price_avg","launch_date","packaging","promotion_flag","ad_channel","premium_flag","competitor_tag","currency"]:
        if col not in df.columns:
            df[col] = None
    df["currency"] = df["currency"].fillna("KRW")

    # features_structured: product_feature를 파싱해서 간단 키워드 리스트(선택)
    def parse_features(x: str) -> List[str]:
        if not isinstance(x, str): return []
        parts = re.split(r"[,/;]\s*", x)
        cleaned = []
        for p in parts:
            p = p.strip()
            p = re.sub(r"[^0-9a-zA-Z가-힣()+\- ]", "", p)
            if p:
                cleaned.append(p[:40])
        return cleaned[:8]
    df["features_structured"] = df["product_feature"].apply(parse_features)

    products = []
    for _, r in df.iterrows():
        products.append({
            "name": r["product_name"],
            "category_level_1": r["category_level_1"],
            "category_level_2": r["category_level_2"],
            "category_level_3": r["category_level_3"],
            "size": r["size"] or "",
            "price": _to_float(r["price_avg"], None),
            "currency": r["currency"] or "KRW",
            "launch_date": str(r["launch_date"]) if pd.notna(r["launch_date"]) else None,
            "packaging": r["packaging"] if pd.notna(r["packaging"]) else None,
            "promotion_flag": bool(r["promotion_flag"]) if pd.notna(r["promotion_flag"]) else None,
            "ad_channel": r["ad_channel"] if pd.notna(r["ad_channel"]) else None,
            "premium_flag": r["premium_flag"] if pd.notna(r["premium_flag"]) else None,
            "competitor_tag": r["competitor_tag"] if pd.notna(r["competitor_tag"]) else None,
            "features_structured": r["features_structured"],
            "raw_feature": r["product_feature"],
        })
    return products

# ===== 4. 페르소나 로드 =====
def load_personas(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "personas" in data:
        return data["personas"]
    if isinstance(data, list):
        return data
    raise ValueError("지원하지 않는 JSON 구조입니다. 최상위가 list 또는 {'personas': [...]} 이어야 합니다.")

### Cell 6: 페르소나 및 제품 정보 요약 함수
- LLM 프롬프트에 들어갈 페르소나와 제품 정보를 간결하게 요약하는 함수

In [21]:
# ===== 5. 페르소나 요약 (수정된 버전) =====
def summarize_persona(p: dict) -> str:
    persona_id = p.get("persona_id", p.get("id", "N/A"))
    seg_id = p.get("segment_id","")
    seg_label = p.get("segment_label","")
    demo = _safe_get(p, ["demographics"], {}) or {}
    shop = _safe_get(p, ["shopping_profile"], {}) or {}
    ori  = _safe_get(p, ["orientations"], {}) or {}

    demo_txt = ", ".join(f"{k}:{v}" for k,v in demo.items() if v not in [None,""])
    
    # --- 수정된 부분 시작 ---
    # None 값이 리스트에 포함된 경우를 대비하여, 문자열로 변환하고 None은 필터링합니다.
    top_categories = shop.get("top_categories") or []
    top_categories_str_list = [str(item) for item in top_categories if item is not None]

    dec_criteria = shop.get("decision_criteria") or []
    dec_criteria_str_list = [str(item) for item in dec_criteria if item is not None]
    # --- 수정된 부분 끝 ---

    pc = shop.get("purchase_cycle") or ""
    ori_txt = ", ".join(f"{k}:{v}" for k,v in ori.items() if v not in [None,""])

    lines = [
        f"persona_id: {persona_id}",
        f"segment_id: {seg_id} / segment_label: {seg_label}",
        f"demographics: {demo_txt}" if demo_txt else "demographics: N/A",
        f"purchase_cycle: {pc}" if pc else "purchase_cycle: N/A",
        f"top_categories: {', '.join(top_categories_str_list) if top_categories_str_list else 'N/A'}",
        f"decision_criteria: {', '.join(dec_criteria_str_list) if dec_criteria_str_list else 'N/A'}",
        f"orientations(1-9): {ori_txt}" if ori_txt else "orientations: N/A",
    ]
    return "\n".join(lines)

def risk_notes(p: dict) -> List[str]:
    ori  = _safe_get(p, ["orientations"], {}) or {}
    notes = []
    if _to_float(ori.get("brand_loyalty"), 0) >= 7:
        notes.append("브랜드 충성 높음 → 기존 익숙한 SKU 고집으로 신제품 전환 낮음(프로모션 시 완화)")
    if _to_float(ori.get("price_sensitivity"), 0) >= 7:
        notes.append("가격 민감 높음 → 평균가 대비 가격상승에 민감(프로모션/대용량에 반응)")
    if _to_float(ori.get("hmr"), 0) >= 7:
        notes.append("HMR 성향 높음 → 간편식/즉석조리 카테고리 적합도↑")
    if _to_float(ori.get("health"), 0) >= 7:
        notes.append("건강/영양 성향 높음 → 저첨가/클린라벨/고단백/저나트륨 속성 선호")
    return notes or ["명시적 리스크 없음(기본 검증 규칙 유지)"]

# ===== 6. 제품 블록 =====
def products_block(products: List[Dict[str,Any]]) -> str:
    rows = []
    for i, prd in enumerate(products):
        rows.append(
            f"- {i+1}. {prd['name']} ({prd.get('size','')}) | "
            f"카테고리:{prd.get('category_level_1','')}/{prd.get('category_level_2','')}/{prd.get('category_level_3','')} | "
            f"예상소비자가:{prd.get('price','N/A')} {prd.get('currency','KRW')} | "
            f"프리미엄:{prd.get('premium_flag','N/A')} | 프로모션:{prd.get('promotion_flag','N/A')}"
        )
    return "\n".join(rows)

### Cell 7: 최적 프롬프트 생성 함수 (핵심 로직)
- 모든 정보를 취합하여 LLM에게 전달할 최종 프롬프트를 생성

In [22]:
# ===== 7. 최적 프롬프트 생성 =====
def build_optimal_prompt(
    persona: dict,
    products: List[Dict[str,Any]],
    horizon_months: int = 12,
    market_context: str = "대한민국 가공식품 소매시장(온·오프라인 합산)",
    base_rate_hint: str = "신제품은 초기 1~3개월 프로모션 효과 후 점차 정상화되는 경향.",
    cap_prob_low: float = 0.02,
    cap_prob_high: float = 0.60
) -> str:
    # 페르소나 요약/리스크
    persona_summary = summarize_persona(persona)
    risks = risk_notes(persona)

    # 월빈도 추정(LLM이 참고하도록 힌트 제공)
    pc_text = _safe_get(persona, ["shopping_profile","purchase_cycle"], "")
    monthly_freq_est = purchase_cycle_to_monthly_freq(pc_text, fallback=1.0)

    # 성향 정규화 값(0~1)
    ori = _safe_get(persona, ["orientations"], {}) or {}
    ori_norm = {
        "health": _norm_1to9(ori.get("health")),
        "price_sensitivity": _norm_1to9(ori.get("price_sensitivity")),
        "hmr": _norm_1to9(ori.get("hmr")),
        "brand_loyalty": _norm_1to9(ori.get("brand_loyalty")),
        "premium": _norm_1to9(ori.get("premium")),
        "convenience": _norm_1to9(ori.get("convenience")),
        "variety_seeking": _norm_1to9(ori.get("variety_seeking")),
    }

    prompt = dedent(f"""
    # 역할
    당신은 **소비자 페르소나 기반 수요예측 애널리스트**입니다. 내부 사고는 메모로만 사용하고, 최종 출력은 지정된 JSON 포맷으로만 반환합니다.

    # 과업
    아래 **페르소나**가 출시 후 {horizon_months}개월 동안 다음 **제품 후보** 각각을 구매할 **월별 확률(0~1)**과 **월별 예상 구매수량(개)**을 추정하세요.
    예산·채널·빈도 제약, 대체·보완 관계, 프로모션/계절성을 반영하고, 극단값을 회피하여 현실적인 분포를 유지하세요.

    # 페르소나(요약)
    {persona_summary}

    # 전처리 힌트(모델 내부 메모용)
    - purchase_cycle 월빈도 추정값: ~{monthly_freq_est:.2f}/월
    - orientations 정규화(0~1): {ori_norm}

    # 리스크·편향 유의
    {', '.join(risks)}

    # 시장 컨텍스트
    {market_context}
    - 베이스레이트 힌트: {base_rate_hint}

    # 제품 후보
    {products_block(products)}

    # 모델링 규칙
    1) **예산/빈도 제약**: 월 예산과 기존 소비주기 준수. 총지출이 비현실적으로 커지지 않도록 수량/확률 동시 조정.
    2) **대체·보완**: 유사 카테고리 간 경쟁/보완성 반영(동시구매/대체 확률).
    3) **프로모션·계절성**: 출시 1~3개월 판촉, 명절/휴가철·계절성, 용량/단위가격 탄력성 반영.
    4) **채널적합성/패키징**: 페르소나의 선호 채널/패키징과의 적합도 반영.
    5) **교정(Calibration)**: 월별 구매확률을 {cap_prob_low}~{cap_prob_high} 범위 중심으로 유도(0/1 극단 회피).
    6) **검증**: 총지출·최대지출·월빈도·재고누적(가정) 체크 후 이상치 수정.

    # 출력 포맷(JSON만 반환, 추가 텍스트 금지)
    {{
      "persona_id": {persona.get('persona_id', persona.get('id', 'null'))},
      "assumptions": {{
        "seasonality": "문장",
        "promotion": "문장",
        "substitution_rules": "문장",
        "budget_binding": true/false
      }},
      "forecast": [
        {{
          "product": "제품명",
          "persona_fit_score": 0.0,
          "months": [
            {{
              "month_index": 1,
              "purchase_prob": 0.0,
              "expected_qty": 0,
              "drivers": ["최대 3개"],
              "barriers": ["최대 3개"],
              "expected_spend_KRW": 0
            }}
            "... {horizon_months}개월까지 반복 ..."
          ]
        }}
        "... 모든 제품 반복 ..."
      ],
      "sanity_checks": {{
        "avg_monthly_spend_KRW": 0,
        "max_single_month_spend_KRW": 0,
        "exceeds_budget": true/false,
        "notes": ["조정/수정 근거"]
      }}
    }}

    # 절차
    - (내부 메모) 제약/가정 정리 → 월별 초기치 → 베이스레이트/예산 교정 → 대체/계절성 반영 → 검증/수정.
    - 최종 응답에는 **JSON만** 포함하세요(설명 금지).
    """).strip()

    return prompt

### Cell 8: 스크립트 실행 및 데모 출력
- 정의된 함수들을 사용하여 실제로 데이터를 불러오고, 첫 번째 페르소나에 대한 프롬프트를 생성하여 화면에 출력

In [23]:
# ===== 8. 실행 예시 / 배치 생성 =====

# 데이터 로드
try:
    personas = load_personas(PERSONA_JSON)
except Exception as e:
    print(f"페르소나 파일('{PERSONA_JSON}') 로드 오류:", e)
    personas = []

try:
    products = load_products(PRODUCTS_CSV)
except Exception as e:
    print(f"제품 파일('{PRODUCTS_CSV}') 로드 오류:", e)
    products = []


# 데모: 앞의 1명에 대해 프롬프트 출력
if personas and products:
    demo_prompt = build_optimal_prompt(personas[0], products, horizon_months=12)
    print("\n===== DEMO PROMPT (1명) =====\n")
    print(demo_prompt)
else:
    print("\n데이터 파일 로드에 실패하여 데모 프롬프트를 생성할 수 없습니다. Cell 2의 파일 경로를 확인해주세요.")

# (옵션) 전체 배치 저장 — 필요 시 아래 코드의 주석을 해제하고 실행하세요.

if personas and products:
    out_path = f"prompts_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for p in personas:
            prompt = build_optimal_prompt(p, products, horizon_months=12)
            rec = {"persona_id": p.get("persona_id", p.get("id")), "prompt": prompt}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print("\n===== BATCH SAVE COMPLETE =====")
    print("저장 완료:", out_path)


===== DEMO PROMPT (1명) =====

# 역할
    당신은 **소비자 페르소나 기반 수요예측 애널리스트**입니다. 내부 사고는 메모로만 사용하고, 최종 출력은 지정된 JSON 포맷으로만 반환합니다.

    # 과업
    아래 **페르소나**가 출시 후 12개월 동안 다음 **제품 후보** 각각을 구매할 **월별 확률(0~1)**과 **월별 예상 구매수량(개)**을 추정하세요.
    예산·채널·빈도 제약, 대체·보완 관계, 프로모션/계절성을 반영하고, 극단값을 회피하여 현실적인 분포를 유지하세요.

    # 페르소나(요약)
    persona_id: 1
segment_id: 2 / segment_label: 프리미엄+편의성선호형
demographics: age:52, gender:여자, region:부산, household:1인 가구, income_bracket:100-200만원 미만, job:단순노무 종사자, marital_status:미혼(사별/이혼 포함), dual_income:False
purchase_cycle: 2주일에 1회
top_categories: 간편식(HMR, 즉석조리, 즉석섭취, 밀키트, 신선편의, 커피 및 차(커피, 커피음료, 잎차, 티백, 곡물차 등
decision_criteria: 품질, 안전성
orientations(1-9): health:7, price_sensitivity:6, hmr:5, brand_loyalty:9, premium:7, convenience:8, variety_seeking:6

    # 전처리 힌트(모델 내부 메모용)
    - purchase_cycle 월빈도 추정값: ~2.15/월
    - orientations 정규화(0~1): {'health': 0.75, 'price_sensitivity': 0.625, 'hmr': 0.5, 'brand_loyalty': 1.0, 'premium': 0.75, 'convenience': 0.875, 'variety_seeking': 0